⏱️ **Reading time:** ~10 minutes | **Type:** Architecture overview (no code)

# 🏗️ RideFlow — Data Mesh Reference Architecture

**A production-grade, multi-domain Data Mesh** built on declarative Data Contracts, powered by [LakeLogic](https://github.com/lakelogic).

This notebook series demonstrates how a global ride-sharing platform (**RideFlow**) organises its data estate as a decentralised Data Mesh — with autonomous domain teams, governed data products, and cross-domain data sharing.

*RideFlow is a fictitious company. All data, metrics, and infrastructure names are synthetic — designed to reflect real-world scale and complexity.*

> **This is Notebook 07a** — the architectural overview. Run the subsequent notebooks (`07b` → `07i`) to see each layer in action.

---
## 1 · The Business: RideFlow

RideFlow is a global mobility platform operating across **6 cities** (London, New York, Berlin, Paris, Tokyo, Sydney). It processes:

| Metric | Scale |
| :-- | :-- |
| **Daily completed trips** | ~250,000 |
| **Active riders** | 4.2M |
| **Active drivers** | 380K |
| **Payment transactions/day** | ~300,000 (Stripe) |
| **Support tickets/day** | ~12,000 (Zendesk) |
| **Marketing channels** | Google Ads, Meta Ads, HubSpot, Google Analytics |

### Why Data Mesh?

RideFlow's data was originally managed by a **centralised data engineering team** — the classic bottleneck:

- The Marketplace team waited weeks for pipeline changes
- The Finance team couldn't reconcile payments without coordinating with 3 other teams
- Marketing built shadow pipelines in Google Sheets
- A single GDPR request took days to execute across disconnected systems
- Backfilling historical data was a nightmare — no one knew which pipelines to replay, in what order, or whether downstream tables would break. A 3-day backfill became a 2-week cross-team coordination exercise

**Data Mesh solves this** by treating data as a **product**, owned by the domain teams who understand it best. Each domain can independently backfill, replay, and version its own data products without impacting other teams — because contracts define the boundaries, and Delta Lake provides time-travel guarantees.

---
## 2 · Data Mesh Principles at RideFlow

RideFlow implements the four foundational Data Mesh principles:

### 🏢 Domain Ownership

Each business domain owns its data end-to-end — from ingestion to published data products:

```
┌─────────────────────────────────────────────────────────────────────────┐
│                        RideFlow Data Mesh                              │
│                                                                         │
│  ┌─────────────────┐  ┌─────────────────┐  ┌─────────────────┐         │
│  │   Marketplace   │  │    Payments      │  │   Operations    │         │
│  │   Domain        │  │    Domain        │  │   Domain        │         │
│  │                 │  │                  │  │                 │         │
│  │  Owner:         │  │  Owner:          │  │  Owner:         │         │
│  │  Marketplace    │  │  Finance         │  │  Trust &        │         │
│  │  Engineering    │  │  Engineering     │  │  Safety         │         │
│  │                 │  │                  │  │                 │         │
│  │  System:        │  │  System:         │  │  Systems:       │         │
│  │  rideflow       │  │  stripe          │  │  zendesk,       │         │
│  │                 │  │                  │  │  checkr, twilio │         │
│  └────────┬────────┘  └────────┬─────────┘  └────────┬────────┘         │
│           │                    │                     │                  │
│           ▼                    ▼                     ▼                  │
│  ┌─────────────────────────────────────────────────────────────┐        │
│  │              Published Data Products (Silver/Gold)          │        │
│  │   silver_rideflow_trips  ·  silver_stripe_charges          │        │
│  │   gold_trip_kpis  ·  gold_payment_reconciliation           │        │
│  │   gold_rider_ltv  ·  gold_surge_pricing_model              │        │
│  └─────────────────────────────────────────────────────────────┘        │
│                                                                         │
│  ┌─────────────────┐                                                   │
│  ┌─────────────────┐  ┌─────────────────┐                             │
│  │   Marketing     │  │   _reference     │                             │
│  │   Domain        │  │   Domain         │                             │
│  │                 │  │                  │                             │
│  │  Owner:         │  │  Static dims:    │                             │
│  │  Growth Team    │  │  cities, vehicle │                             │
│  │                 │  │  types, surge    │                             │
│  │  Systems:       │  │  tiers, dim_date │                             │
│  │  GA, HubSpot,   │  └─────────────────┘                             │
│  │  Meta, Google   │                                                   │
│  └─────────────────┘                                                   │
│                                                                         │
│  ┌──────────────────────────────────┐  ┌──────────────────────────┐    │
│  │  Federated Governance Layer      │  │  _shared Domain          │    │
│  │  Data Contracts · SLOs · GDPR   │  │  Cross-domain Gold       │    │
│  │  Cost Controls · Lineage        │  │  marketplace_health,     │    │
│  └──────────────────────────────────┘  │  revenue_daily,          │    │
│                                        │  driver_360              │    │
│                                        └──────────────────────────┘    │
└─────────────────────────────────────────────────────────────────────────┘
```

### 📦 Data as a Product

Each domain publishes **curated, documented, quality-checked outputs** — not raw internal tables. Consumers read from Silver/Gold data products with guaranteed SLAs.

### 🏗️ Self-Serve Data Platform

LakeLogic provides the **self-serve infrastructure** — any domain team can define a YAML contract, run a pipeline, and publish a data product without writing boilerplate ETL code.

### 🔐 Federated Computational Governance

Quality rules, SLOs, cost budgets, and privacy controls are **declared in contracts** and **enforced automatically for each domain** — not by a central gatekeeper.

---
## 3 · High-Level Architecture

### Medallion Architecture (per Domain)

Each domain runs an independent Medallion pipeline, governed by its own contracts:

```
  Landing Zone         Bronze              Silver              Gold
  ┌──────────┐        ┌──────────┐        ┌──────────┐       ┌──────────┐
  │ CSV/JSON  │──────▶│ Raw      │──────▶│ Typed    │──────▶│ KPIs     │
  │ Parquet   │ ingest│ Append   │ clean │ Deduped  │ agg  │ Models   │
  │ API pulls │       │ Schema   │       │ Merged   │       │ Products │
  └──────────┘        │ Enforced │        │ FK Valid │       └──────────┘
                      └──────────┘        └──────────┘
                           │                   │
                      ┌────▼───────────────────▼────┐
                      │     Quarantine Zone         │
                      │  (contract-violating rows)  │
                      └─────────────────────────────┘
```

### Cross-Domain Data Flow

Domains **consume each other's data products** — never internal tables:

```
  Marketplace Domain                     Payments Domain
  ┌─────────────────────┐               ┌─────────────────────────┐
  │ Bronze → Silver     │               │ Bronze → Silver         │
  │                     │               │                         │
  │ DATA PRODUCT:       │  trip_id      │ Reads Marketplace       │
  │ silver_rideflow_    │──────────────▶│ data product to         │
  │ trips               │               │ reconcile charges       │
  │                     │               │                         │
  │ DATA PRODUCT:       │               │ DATA PRODUCT:           │
  │ gold_trip_kpis      │               │ gold_payment_reconcil.  │
  └─────────────────────┘               └─────────────────────────┘
           │                                       │
           │         Operations Domain             │
           │         ┌───────────────────┐         │
           │         │ Reads driver      │         │
           └────────▶│ profiles to       │         │
                     │ enrich tickets    │         │
                     └───────────────────┘         │
           │                                       │
           ▼              ▼                        ▼
  ┌────────────────────────────────────────────────────────┐
  │            _shared: Cross-Domain Gold Products         │
  │  marketplace_health · revenue_daily · driver_360      │
  └────────────────────────────────────────────────────────┘
```

### 🧠 Advanced Data Synthesis: Time-Aware & Edge Cases

To ensure RideFlow's dashboards and mesh observability reflect a living business, the synthetic data isn't just random noise - it is structured to simulate realistic market behaviors and contract violations.

**1. Temporal & Geographic Market Profiles**
Synthetic data is wrapped in business context before it hits the landing zone:
- **Time Profiles**: Adjusts ride volume probabilities to create **Morning Peaks** (07:00-09:00, 2.5x), **Evening Peaks** (17:00-20:00, 3.0x), and **Friday Night Surge Windows** (20:00-02:00, 5.0x).
- **City Profiles**: Hardcodes accurate timezones and scaling factors (e.g., `NYC` base volume is 800 trips/hr vs `BER` 200 trips/hr).

**2. Quarantine Injection Targeting (TC-001 to TC-008)**
For our core `trip_completed` entity, the generator explicitly injects **exactly 800 edge-case rows** covering 8 specific failure modes:

| TC | Description | Rows | What the pipeline catches |
| :-- | :-- | --: | :-- |
| TC-001 | Negative fare | 120 | Business logic: `fare_amount >= 0` |
| TC-002 | driver_id = rider_id | 80 | Self-service fraud detection |
| TC-003 | Dropoff before pickup | 150 | Temporal sanity: `dropoff_at > pickup_at` |
| TC-004 | Distance > 500km | 100 | Geospatial bounds check |
| TC-005 | SQL injection in notes | 150 | Security / input sanitization |
| TC-006 | Surge > 5.0x | 80 | Cap violation: `surge_multiplier <= 5.0` |
| TC-007 | Unknown city code | 60 | Referential integrity check |
| TC-008 | Duplicate trip_id | 60 | Idempotency / uniqueness constraint |

**3. Hybrid Demo Strategy**
The Marketplace pipeline (`07b`) uses a two-phase approach for maximum impact:

1. **Phase 1 - Streaming Simulation**: The `StreamingSimulator` generates 24 hours of organic, time-curved ride data. The dashboard animates in real-time as micro-batches arrive - this is the visual "wow" moment.
2. **Phase 2 - Deterministic Injection**: Exactly 800 edge-case rows (TC-001 to TC-008) are surgically injected into the landing zone before Bronze ingestion begins.

When the pipeline processes both phases together, the quarantine dashboard shows **100% reconciliation**:
```
source_count = good_count + bad_count
"Every row accounted for."
```
This hybrid approach gives you both a living, breathing dashboard *and* mathematically precise failure-mode coverage.

---
## 4 · The Role of LakeLogic

LakeLogic is the **self-serve data platform layer** that makes the Data Mesh operationally viable. Without it, each domain team would need to write bespoke ETL, quality checks, and materialization logic.

### What LakeLogic Provides

| Capability | How It Works |
| :-- | :-- |
| **Declarative Contracts** | Define schema, quality rules, and lineage in YAML — zero Python/SQL boilerplate |
| **Pipeline Runner** | `runner.run(target_layers="bronze,silver,gold")` — one method for the full Medallion pipeline |
| **Quality Enforcement** | Row-level and dataset-level rules quarantine bad data automatically |
| **Schema Evolution** | Contract changes are tracked; unknown fields are quarantined or merged based on policy |
| **Materialization** | Write to Delta Lake with merge/append/overwrite strategies |
| **Run Logging** | Every pipeline run is audited: row counts, durations, quarantine ratios, estimated costs |
| **GDPR Erasure** | `_execute_gdpr_pass()` scans all contract-governed tables and redacts subject data |
| **Multi-Engine** | Prototype locally with Polars + DuckDB, deploy the exact same contracts to production with Spark — zero rewrites (see below) |
| **Synthetic Data** | `DataGenerator` and `StreamingSimulator` produce realistic test data from contracts |

### Write Once, Run Anywhere: Local → Production

One of LakeLogic's most powerful features is the ability to **develop and test locally**, then deploy the **exact same contracts and configuration** to production — no code changes required:

```
  LOCAL DEVELOPMENT                          PRODUCTION (e.g., Databricks)
  ┌────────────────────────┐                ┌────────────────────────┐
  │  Engine: Polars        │                │  Engine: Spark         │
  │  Catalog: DuckDB       │  same YAML     │  Catalog: Unity Catalog│
  │  Storage: ./lakehouse  │ ─────────────▶ │  Storage: ADLS Gen2    │
  │  Cost: £0              │  same contract │  Cost: DBU-metered     │
  └────────────────────────┘                └────────────────────────┘
```

This means a data engineer can:
1. **Prototype** a new contract on their laptop using Polars — the entire Marketplace pipeline (7 Bronze + 4 Silver + 4 Gold tables) runs in **under 30 seconds** locally, with zero cloud costs or cluster spin-up
2. **Validate** quality rules and schema enforcement against local DuckDB tables
3. **Deploy** to a production engine (e.g., Databricks) with `engine="spark"` — the contracts, quality rules, and materialization strategies are identical

The environment block in `_system.yaml` is the only thing that changes between contexts:

```yaml
environments:
  local:                         # Laptop / CI
    catalog: "local"
    storage_root: "./lakehouse"
  dev:                           # Cloud dev
    catalog: "rideflow-lakehouse-dev-001"
  prod:                          # Production
    catalog: "rideflow-lakehouse-prod-001"
```

### The Three-Layer Configuration Model

LakeLogic organises configuration into three nested layers:

```
  ┌───────────────────────────────────────────────────────────┐
  │  1. DOMAIN  (_domain.yaml)                               │
  │     Ownership, SLOs, cost budgets, schedule              │
  │     Example: marketplace._domain.yaml                    │
  │     "Marketplace Engineering owns this, 99% quality SLO" │
  │                                                          │
  │  ┌───────────────────────────────────────────────────┐    │
  │  │  2. SYSTEM  (_system.yaml)                       │    │
  │  │     Storage paths, environments, materialization  │    │
  │  │     Example: marketplace/rideflow/_system.yaml    │    │
  │  │     "Write to ADLS in dev, local disk in local"   │    │
  │  │                                                   │    │
  │  │  ┌───────────────────────────────────────────┐    │    │
  │  │  │  3. CONTRACT  (entity_v1.0.yaml)         │    │    │
  │  │  │     Schema, quality rules, transformations│    │    │
  │  │  │     Example: bronze_rideflow_trips_v1.0   │    │    │
  │  │  │     "trip_id is string, required, unique" │    │    │
  │  │  └───────────────────────────────────────────┘    │    │
  │  └───────────────────────────────────────────────────┘    │
  └───────────────────────────────────────────────────────────┘
```

This separation means:
- **Domain-level** concerns (who owns it, what SLOs apply) are set once, not in every contract
- **System-level** concerns (where to store, which environments) are shared across all entities in a system
- **Contract-level** concerns (what fields, what quality rules) are the only thing data engineers need to write

---
## 5 · Domain Inventory

RideFlow's Data Mesh comprises **4 autonomous domains**, each with distinct ownership, SLOs, and source systems:

| Domain | Owner | Systems | Bronze Contracts | Silver Contracts | Gold Contracts | SLO (Quality) |
| :-- | :-- | :-- | :--: | :--: | :--: | :-- |
| **Marketplace** | Marketplace Engineering | rideflow | 7 | 4 | 4 | 99.99% critical |
| **Payments** | Finance Engineering | stripe | 3 | 3 | 1 | 99.99% critical |
| **Operations** | Trust & Safety | zendesk, checkr, twilio | 1+ | 1+ | — | 99.9% critical |
| **Marketing** | Growth Team | GA, HubSpot, Meta, Google Ads | 2+ | 2+ | — | 99% critical |
| **_shared** | Platform / Data Mesh Team | cross-domain | — | — | 3+ | N/A (composed) |
| **_reference** | Platform / Data Mesh Team | _internal | — | — | dim_date, cities, etc. | N/A (static) |

### Marketplace Domain (Core)

The core transactional domain — trips, riders, drivers, telemetry. Operates across 6 cities with surge pricing and real-time GPS feeds.

**Key entities:** `trip_completed`, `trip_requests`, `trip_cancellations`, `rider_profiles`, `driver_profiles`, `driver_telemetry`, `rider_app_events`

**Gold data products:** Trip KPIs, Driver Scorecard, Rider Lifetime Value, Surge Pricing Model (EU AI Act tagged)

### Payments Domain

Financial transactions via Stripe. Reconciles charges against Marketplace trips for revenue reporting.

**Key entities:** `stripe_charges`, `stripe_payouts`, `stripe_refunds`

**Cross-domain dependency:** Reads `silver_rideflow_trips` to match `trip_id → charge_id`

### Operations Domain

Trust & Safety tooling — support tickets (Zendesk), background checks (Checkr), communications (Twilio). A real-world example of one domain team managing multiple SaaS sources.

### Marketing Domain

Growth analytics — web sessions (GA), CRM (HubSpot), ad spend (Meta, Google Ads). Reads Marketplace's Rider LTV for attribution analysis.

### _shared Domain (Cross-Domain Gold)

The platform team's **cross-domain Gold products** — these consume published data products from multiple domains and produce unified mesh-level views. Examples: `marketplace_health`, `revenue_daily`, `driver_360`, `acquisition_cost`. External source dependencies are explicitly declared in `_shared/_system.yaml`.

### _reference Domain (Static Dimensions)

Static reference data that **every domain can depend on**: city codes, vehicle types, surge pricing tiers, cancellation reason codes, and generated dimensions like `dim_date`. These are loaded once and versioned — they don't go through the streaming Medallion pipeline.

---
## 6 · Federated Governance

Governance in LakeLogic is **layered** — domain-level defaults cascade down, but can be overridden at the system or individual contract level. This means domain owners set the baseline, while specific tables can tighten or relax rules as needed.

```
  Scope            Declared In              Can Override?
  ─────────────    ─────────────────────     ─────────────────────────────
  Domain-wide      _domain.yaml              Baseline for all systems
  System-wide      _system.yaml              Overrides domain defaults
  Per-table        entity_v1.0.yaml          Overrides system defaults
```

---

### Service Level Objectives (SLOs)

**What:** Freshness, row-count anomaly detection, quality thresholds, and schedule expectations.  
**Scope:** Declared at the **domain** level in `_domain.yaml`. Applies to every pipeline in the domain.  
**Override:** A system or contract can tighten thresholds (e.g., a critical financial table requiring 99.99% quality vs. the domain default of 99%).

```yaml
# From marketplace/_domain.yaml
slo:
  freshness:
    bronze:
      max_delay_minutes: 30
      check_column: "_lakelogic_loaded_at"
    silver:
      max_delay_minutes: 120
      check_column: "_lakelogic_processed_at"
  row_count:
    bronze:
      anomaly:
        enabled: true
        lookback_runs: 14
        method: "median"
  quality:
    min_good_ratio: 0.99
    by_severity:
      critical:
        min_good_ratio: 0.9999
  schedule:
    expected_start_utc: "04:00"
    expected_completion_utc: "05:00"
```

---

### Cost Governance

**What:** Budget limits (daily, weekly, monthly) and anomaly detection for runaway pipelines.  
**Scope:** Declared at the **domain** level. Each domain team is accountable for their own compute spend.

```yaml
# From marketplace/_domain.yaml
cost:
  currency: "GBP"
  budget:
    daily_limit: 50.00
    weekly_limit: 250.00
    monthly_limit: 800.00
    per_run_anomaly_multiplier: 3.0
    alert_channels: ["slack", "email"]
```

---

### Materialization & Schema Policy

**What:** How data is written (append, merge, overwrite) and how schema changes are handled (strict, allow, quarantine).  
**Scope:** Declared at the **system** level in `_system.yaml`. Individual contracts can override.  
**Override:** A Gold contract might use `overwrite` while the system default for Silver is `merge`.

```yaml
# From marketplace/rideflow/_system.yaml
materialization:
  bronze:
    strategy: append          # Raw data, always append
    format: delta
  silver:
    strategy: merge           # Upsert on primary key
    format: delta
    merge_dedup_guard: true
  gold:
    strategy: merge
    format: delta

server:
  bronze:
    cast_to_string: true      # Ingest everything as string first
    schema_policy:
      evolution: append       # New columns are accepted
      unknown_fields: allow
  silver:
    schema_policy:
      evolution: strict       # Schema changes break the build
      unknown_fields: quarantine
```

---

### Lineage & Audit Trail

**What:** Automatic stamping of every row with source file, processing timestamp, run ID, and contract name.  
**Scope:** Declared at the **system** level. Every table in the system inherits these columns automatically.

```yaml
# From marketplace/rideflow/_system.yaml
lineage:
  enabled: true
  source_column_name: "_lakelogic_source"
  timestamp_column_name: "_lakelogic_processed_at"
  run_id_column_name: "_lakelogic_run_id"
  contract_name_column_name: "_lakelogic_contract_name"
```

---

### Data Quality Rules

**What:** Row-level and dataset-level validation rules. Rows that fail are routed to quarantine — never silently dropped.  
**Scope:** Declared at the **contract** level (per table). This is where domain experts encode their business logic.

```yaml
# From gold_rideflow_surge_pricing_model_v1.0.yaml
quality:
  row_rules:
    - name: valid_surge_bounds
      sql: "surge_multiplier >= 1.0 AND surge_multiplier <= 5.0"
    - name: demand_index_bounds
      sql: "demand_index >= 0.0 AND demand_index <= 10.0"
  dataset_rules:
    - unique: inference_id
```

---

### Environment Management

**What:** Storage paths and catalog names switch per environment — the same contracts work in dev, staging, prod, and locally.  
**Scope:** Declared at the **system** level. Zero code changes between environments.

```yaml
# From marketplace/rideflow/_system.yaml
environments:
  dev:
    catalog: "rideflow-lakehouse-dev-001"
    storage_account: "sarideflowdevadls001"
  prod:
    catalog: "rideflow-lakehouse-prod-001"
    storage_account: "sarideflowprodadls001"
  local:
    catalog: "local"
    storage_root: "./lakehouse"
    data_root: "./lakehouse/marketplace"
```

---

### EU AI Act Compliance

**What:** Regulatory metadata for ML-driven data products (risk tier, bias examination, transparency).  
**Scope:** Declared at the **contract** level — only relevant for Gold tables that contain ML outputs.

```yaml
# From gold_rideflow_surge_pricing_model_v1.0.yaml
compliance:
  eu_ai_act:
    applicable: true
    risk_tier: "high"
    risk_tier_rationale: "Dynamic pricing models determining algorithmic
      economic outcomes under bias scrutiny."
    bias_examination: true
    transparency_disclosure: true
    human_oversight: true
```

---
## 7 · Notebook Series Guide

This project is decomposed into domain-owned notebooks that mirror real-world team boundaries:

### Tier 1 — Domain Pipelines

Each notebook represents what a specific domain team owns and operates.

| Notebook | Domain | What It Does |
| :-- | :-- | :-- |
| **`07b` Marketplace** | Marketplace Eng | Generates ride data, runs Bronze → Silver → Gold, publishes trip/rider/driver data products |
| **`07c` Payments** | Finance Eng | Generates Stripe data, processes charges/refunds, cross-domain joins to Marketplace trips |
| **`07d` Operations** | Trust & Safety | *(Phase 2)* Zendesk tickets, Checkr background checks, Twilio comms |
| **`07e` Marketing** | Growth Team | *(Phase 2)* GA sessions, HubSpot contacts, ad spend attribution |

### Tier 2 — Cross-Domain Products

| Notebook | What It Does |
| :-- | :-- |
| **`07f` Mesh Products** | *(Phase 2)* Reads published data products from all domains, builds unified mesh-level KPIs |

### Tier 3 — Business Scenario Playbooks

| Notebook | What It Does |
| :-- | :-- |
| **`07g` GDPR/RTBF** | Executes a Right-to-Be-Forgotten erasure across all domains with audit report |
| **`07h` Backfill** | *(Phase 2)* Historical backfill, Delta time travel, quarantine remediation |

### Tier 4 — Observability

| Notebook | What It Does |
| :-- | :-- |
| **`07i` Dashboards** | Panel-based live dashboards: mesh health, pipeline observatory, compliance registry |

### Execution Order

```
07a (this notebook — read only)
 │
 ├──▶ 07b Marketplace   ──┐
 │                        ├──▶ 07f Mesh Products
 ├──▶ 07c Payments     ──┘         │
 │                                 ▼
 │                           07i Dashboards
 │
 └──▶ 07g GDPR (run after 07b+07c to have data to erase)
```

---
## 8 · Project Structure

```
lakelogic-ra-rideflow/
│
├── domains_rideflow/                   # Data Contract definitions
│   ├── _shared/                        #   Cross-domain Gold products
│   │   └── _system.yaml                #   External source declarations
│   ├── marketplace/
│   │   ├── _domain.yaml                #   Domain: ownership, SLOs, cost
│   │   └── rideflow/
│   │       ├── _system.yaml             #   System: storage, environments
│   │       └── contracts/
│   │           ├── bronze/              #   7 raw ingestion contracts
│   │           ├── silver/              #   4 cleaned/merged contracts
│   │           └── gold/                #   4 KPI/ML contracts
│   ├── payments/
│   │   ├── _domain.yaml
│   │   └── stripe/
│   │       ├── _system.yaml
│   │       └── contracts/               #   charges, payouts, refunds
│   ├── operations/
│   │   ├── _domain.yaml
│   │   └── zendesk/ checkr/ twilio/
│   └── marketing/
│       ├── _domain.yaml
│       └── google_analytics/ hubspot/ meta_ads/ google_ads/
│
├── lakehouse/                           # Materialised Delta tables
│   ├── marketplace/
│   │   ├── bronze/                      #   Raw partitioned data
│   │   ├── silver/                      #   Typed, deduplicated
│   │   ├── gold/                        #   KPIs, ML models
│   │   ├── _quarantine/                 #   Contract-violating rows
│   │   └── _logs/                       #   Pipeline run audit trail
│   └── payments/                        #   Same structure
│
├── dashboards/                          # Panel dashboard components
├── infra/                               # Terraform (Azure deployment)
├── src/databricks/                      # Databricks DAB bundle
│
├── 07a_rideflow_intro.ipynb             # ← You are here
├── 07b_rideflow_marketplace.ipynb
├── 07c_rideflow_payments.ipynb
├── 07g_compliance_gdpr_rtbf.ipynb
└── 07i_data_mesh_dashboards.ipynb
```

---
## 🚀 Ready to Run?

Open **`07b_rideflow_marketplace.ipynb`** to begin generating data and running the Marketplace domain pipeline.

The entire series can be executed sequentially — each notebook builds on the outputs of the previous ones, exactly as domain teams would operate in production.